# 03 · LoRA SFT of SmolVLM-500M (A100)
Order: smoke test → single run → small grid. Checkpoints are written straight to Drive so a disconnect only loses minutes (re-run with `--resume`).
Runtime → Change runtime type → **A100 GPU** (L4 also works; batch settings adapt to GPU memory).

## 0. Session setup (run every session)
Code is cloned from GitHub (`moghbaie/InspectEdge_Mobile_VLM`); nothing needs to be uploaded.
Google Drive (`MyDrive/InspectEdge`) only stores outputs: reports, manifests, checkpoints.
Runtime → Change runtime type → **GPU** (T4 is enough for 500M; L4/A100 for 3B).

In [ ]:
import os, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO  = 'https://github.com/moghbaie/InspectEdge_Mobile_VLM.git'
DRIVE = '/content/drive/MyDrive/InspectEdge'   # persistent outputs: reports, manifests, checkpoints
WORK  = '/content/InspectEdge'                 # fast local disk: code + raw images
os.makedirs(DRIVE, exist_ok=True)
if os.path.isdir(f'{WORK}/.git'):
    !git -C {WORK} pull -q --ff-only
else:
    !git clone -q {REPO} {WORK}
assert os.path.exists(f'{WORK}/pyproject.toml'), 'clone failed: check REPO URL / repo visibility'
%cd {WORK}
!git log --oneline -1
!pip -q install -r requirements-colab.txt
!pip -q install -e . --no-deps
os.environ['INSPECTEDGE_ROOT'] = WORK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, transformers, peft
assert transformers.__version__ == '4.57.1', 'pinned transformers not active: Runtime -> Restart session, then re-run this cell'
print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| peft', peft.__version__)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'

In [ ]:
# Data: official VisA release (CC BY 4.0). Verified by SHA-256 before use.
VISA_SHA = '2eb8690c803ab37de0324772964100169ec8ba1fa3f7e94291c9ca673f40f362'
if not os.path.exists('data/raw/visa/pcb1'):
    os.makedirs('data/raw/_dl', exist_ok=True); os.makedirs('data/raw/visa', exist_ok=True)
    !wget -q -O data/raw/_dl/VisA_20220922.tar https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar
    h = subprocess.run(['sha256sum', 'data/raw/_dl/VisA_20220922.tar'], capture_output=True, text=True).stdout.split()[0]
    assert h == VISA_SHA, f'VisA archive checksum mismatch: {h}'
    !tar -xf data/raw/_dl/VisA_20220922.tar -C data/raw/visa && rm data/raw/_dl/VisA_20220922.tar
!ls data/raw/visa

In [ ]:
# Manifest: rebuilt deterministically each session; fails loudly if the locked split changed.
!python -m inspectedge.data.build_manifest

In [ ]:
# Ampere+ GPU (A100 preferred; L4/H100 also fine). bf16 + TF32 need compute capability >= 8.0.
gpu = torch.cuda.get_device_name(0); mem_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability(0)[0] >= 8, \
    f'{gpu} lacks bf16: Runtime -> Change runtime type -> A100 (or L4) GPU'
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
print(gpu, f'{mem_gb:.0f} GB')

In [ ]:
# Memory-sized overrides. Effective batch stays 16 (= T4 config 4x4) so lr/warmup in configs/train_vlm.yaml remain valid.
if mem_gb > 60:   BS, CKPT_ = 16, False   # A100/H100 80GB
elif mem_gb > 30: BS, CKPT_ = 8, False    # A100 40GB
else:             BS, CKPT_ = 8, True     # L4 24GB
GPU_SET = f'batch_size={BS} grad_accum={16 // BS} grad_ckpt={str(CKPT_).lower()} workers={min(8, os.cpu_count())} tf32=true'
print(GPU_SET)

## 1. Training tests: one-batch overfit smoke test (must print SMOKE TEST PASSED)

In [ ]:
!python scripts/train_vlm_lora.py --smoke --set output_dir={DRIVE}/checkpoints {GPU_SET} 2>&1 | grep -vE 'Warning|warn' | tail -15

## 2. Main run (bf16; on OOM append `grad_ckpt=true`, or halve `batch_size` and double `grad_accum`)

In [ ]:
!python scripts/train_vlm_lora.py --resume --set output_dir={DRIVE}/checkpoints {GPU_SET} 2>&1 | grep -vE 'Warning|warn' | tail -30

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
meta = json.load(open(f'{DRIVE}/checkpoints/smolvlm500m_lora_r16_lr2e-4_s0/run_meta.json'))
import matplotlib.pyplot as plt; plt.plot(meta['losses']); plt.title('train loss'); plt.show()
print(meta['gpu'], round(meta['train_seconds']/60,1), 'min')

## 3. Grid (plan 5.4) — run across sessions if needed; each run resumable

In [ ]:
GRID = [(1e-4, 8), (1e-4, 16), (2e-4, 8)]   # (2e-4,16) is the main run above
for lr, r in GRID:
    name = f'smolvlm500m_lora_r{r}_lr{lr:g}_s0'
    !python scripts/train_vlm_lora.py --resume --set output_dir={DRIVE}/checkpoints {GPU_SET} run_name={name} lr={lr} lora_r={r} lora_alpha={2*r} 2>&1 | tail -3

## 4. Ablations (plan 5.5) — same eval harness in notebook 04; always keep `{GPU_SET}` in `--set`
- clean-only: `--set aug_prob=0.0 run_name=..._cleanonly`
- no abstention/mismatch: `--set mismatch_frac=0.0`
- PCB-only MVP: `--set categories=[pcb1,pcb2,pcb3,pcb4]`
- second seed for finalists: `--set seed=1 run_name=..._s1`

In [ ]:
# Persist outputs to Drive (run at the end and after long steps)
!rsync -aR reports data/manifests {DRIVE}/
print('synced to', DRIVE)